# IMDB???????????

????:
- [Model](#Model-Part)
- [DataLoader](#DataLoader-Part)
- [Train_loop & Test_loop](#Train_loop-Part)
- [Controller](#Controller-Part)
- [Console](#Console-Part)
- [Evaluater](#Evaluater-Part)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch
import pandas as pd
import torch.nn as nn
import torch.nn.functional as F
from typing import Callable
from pathlib import Path
import torch.optim as optim
import math
from torch.utils.data import DataLoader, Dataset
DATAPATH = '/kaggle/input/competitions/sjtu-2026-spring-lstm'

## `Model` Part

### LSTM

- `nn.LSTM`: ??????????????(LSTM)????????????????????,?????????:
  $$
  \begin{aligned}
   i_t &= \sigma(W_{\text{ii}}x_t + b_{\text{ii}} + W_{\text{hi}}h_{t-1} + b_{\text{hi}}) \\
   f_t &= \sigma(W_{\text{if}}x_t + b_{\text{if}} + W_{\text{hf}}h_{t-1} + b_{\text{hf}}) \\
   g_t &= \tanh(W_{\text{ig}}x_t + b_{\text{ig}} + W_{\text{hg}}h_{t-1} + b_{\text{hg}}) \\
   o_t &= \sigma(W_{\text{io}}x_t + b_{\text{io}} + W_{\text{ho}}h_{t-1} + b_{\text{ho}}) \\
   \color{orange}{c_t} \, &\color{orange}{=f_t \odot c_{t-1} + i_t \odot g_t} \\
   h_t &= o_t \odot \tanh(c_t)
   \end{aligned}
   $$
   ??:
   - $h_t$ ????? $t$ ?????;
   - $c_t$ ????? $t$ ??????(cell state);
   - $x_t$ ???? $t$ ???;
   - $h_{t-1}$ ????????????,??????????(? $t = 0$ ?);
   - $i_t, f_t, g_t, o_t$ ???????????????????;
   - $\sigma$ ? sigmoid ????;
   - $\odot$ ?? Hadamard(???)???

- `LSTM`: ???`lstm`??????????`FNN`??

   `forward`: 
   $$
   \begin{aligned}
   \text{output}, \textcolor{orange}{hidden},\text{cell} = \text{lstm}(\text{input}) \\
   \text{probs} = \text{Softmax}(\text{FNN}(\textcolor{orange}{hidden}))
   \end{aligned}
   $$

In [ ]:
class LSTM(nn.Module):
    def __init__(self, 
                vocab_size : int, 
                embedding_dim : int, 
                hidden_dim : int, 
                n_layers:int = 1,
                dropout:float = 0.2):
        super(LSTM, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.lstm = nn.LSTM(embedding_dim, 
                            hidden_dim, 
                            num_layers = n_layers, 
                            batch_first=True, 
                            dropout=dropout if n_layers > 1 else 0
                            )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim, 2) # ?????
    
    def forward(self, x):
        embedded = self.dropout(self.embedding(x))
        _, (hidden_t, _) = self.lstm(embedded)
        logits = self.fc(self.dropout(hidden_t[-1, :, :]))
        return logits


## `DataLoader` Part

- `IMDBDataset`:?????`Pytorch`???`Dataset`?????,????????????(??`Dataset`???????`__getitem__`??,?????`__len__`????????????)

`prepare_data_loader`: ?????????????????????????,???????????,????????????????????`PyTorch`?`DataLoader`??????,??????????????:  
- ??????????
- ???????????????????
- ??????????`Dataset`??
- ???????,?????????????

?????????????????({"train": `train_loader`, "test": `test_loader`})

In [ ]:
class IMDBDataset(Dataset):
    def __init__(self, x, y):
        self._x = x
        self._y = y

    def __getitem__(self, idx):
        return {
            "encoded_indices": torch.tensor(self._x[idx], dtype=torch.long),
            "label": torch.tensor(self._y[idx], dtype=torch.long),
        }

    def __len__(self):
        return len(self._x)


def prepare_data_loader(
    path: str,
    ratio: float,
    batch_size: int,
    num_workers: int = 4,
) -> dict:
    """
    ??:
        path (str): .npz??????????
        ratio (float, optional): ?????????(???0.8)
        seed (int, optional): ?????????(???12)
        batch_size (int, optional): ??????(???128)
        num_workers (int, optional): ??????????(???4)

    ??:
        dict: ???????????????
    """
    train_data = np.load(path)

    x_data = train_data["x_train"]
    y_data = train_data["y_train"]

    num_samples = len(x_data)
    split_idx = int(num_samples * ratio)
    x_train = x_data[:split_idx]
    y_train = y_data[:split_idx]
    x_test = x_data[split_idx:]
    y_test = y_data[split_idx:]

    train_batch_size = batch_size
    test_batch_size = train_batch_size

    # ??PyTorch???
    train_dataset = IMDBDataset(x_train, y_train)
    test_dataset = IMDBDataset(x_test, y_test)

    # ???????
    train_loader = DataLoader(
        dataset=train_dataset,
        batch_size=train_batch_size,
        shuffle=True,
        num_workers=num_workers,
        drop_last=True,
    )

    test_loader = DataLoader(
        dataset=test_dataset,
        batch_size=test_batch_size,
        shuffle=False,
        num_workers=num_workers,
    )

    return {"train": train_loader, "test": test_loader}

## `Train_loop` Part
??????????????????????????:
- ??????????,?????????
- ??????????(Metric),????????????

?????`train_step`?`eval_step`??????????(step)??????????,`train_per_epoch`?`test_per_epoch`????????????(epoch)???????????????????,??`step`?????(`batch`)?????,??`epoch`????`step`,??????????????,??????`epoch`,??????????`batch`?????,?:
$$ \text{batch\_sizes} * \text{num\_steps} = \text{total\_data} $$

`train_per_epoch` : ?????????`epoch`?????:
- ?????????
- ???????????????
- ??????????????????????????
- ??????????????

`test_per_epoch` : ?????????`epoch`????????:
- ?????????
- ???????????????
- ????????????????
- ??????(???????),?????????


In [ ]:
from datetime import datetime
import time

def train_step(
    model: nn.Module, 
    loss_fn: Callable, 
    optimizer: optim.Optimizer, 
    batch: dict, 
    device: torch.device
):
    model.train()
    optimizer.zero_grad()

    batch_tokens = batch["encoded_indices"].to(device)
    labels = batch["label"].to(device)

    logits = model(batch_tokens)
    loss = loss_fn(logits, labels)

    loss.backward()
    optimizer.step()

    _, predicted = torch.max(logits, 1)
    correct = (predicted == labels).sum().item()
    total = labels.size(0)

    return loss.item(), correct, total


def eval_step(model: nn.Module, metric_fn: Callable, batch: dict, device: torch.device):
    model.eval()
    with torch.no_grad():
        batch_tokens = batch["encoded_indices"].to(device)
        labels = batch["label"].to(device)
        logits = model(batch_tokens)
        metric = metric_fn(logits, labels)

        return metric.item(), logits, labels


def train_per_epoch(
    model: nn.Module,
    loss_fn: Callable,
    optimizer: optim.Optimizer,
    train_loader: DataLoader,
    device: torch.device,
    epoch: int,
    start_time: float,
    global_batch_counter: int,
    history: dict,
):
    """?? (epoch_avg_loss, epoch_acc, updated_global_batch_counter)"""
    model.train()
    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    for batch_idx, batch in enumerate(train_loader):
        loss, correct, n = train_step(model, loss_fn, optimizer, batch, device)
        total_loss += loss
        total_correct += correct
        total_samples += n
        global_batch_counter += 1

        # ????batch?loss?acc
        history["train_loss_per_batch"].append(loss)
        history["train_acc_per_batch"].append(correct / n)
        history["batch_indices"].append(global_batch_counter)

        # ?5???????
        if (batch_idx + 1) % 5 == 0:
            elapsed = time.time() - start_time
            batch_acc = correct / n
            print(f"Epoch {epoch+1}, Batch {batch_idx+1}, "
                  f"Time: {elapsed:.1f}s, "
                  f"Train Loss: {loss:.4f}, Train Acc: {batch_acc*100:.1f}%")

    epoch_avg_loss = total_loss / len(train_loader)
    epoch_acc = total_correct / total_samples
    return epoch_avg_loss, epoch_acc, global_batch_counter


def test_per_epoch(
    model: nn.Module,
    metric_fn: Callable,
    test_loader: DataLoader,
    device: torch.device,
):
    """?? (avg_loss, accuracy)"""
    model.eval()
    total_loss = 0.0
    correct = 0
    num_batches = len(test_loader)
    num_data = len(test_loader.dataset)

    with torch.no_grad():
        for batch in test_loader:
            loss, logits, labels = eval_step(model, metric_fn, batch, device)
            total_loss += loss
            _, predicted = torch.max(logits, 1)
            correct += (predicted == labels).sum().item()

    avg_loss = total_loss / num_batches
    accuracy = correct / num_data
    return avg_loss, accuracy
    
def plot_metrics(history, epoch):
    """?5?epoch???loss?acc??,???????"""
    batch_x = history["batch_indices"]

    # --- Loss ? ---
    fig, ax = plt.subplots(1, 1, figsize=(8, 4))
    ax.plot(batch_x, history["train_loss_per_batch"], label="Train Loss", alpha=0.6)
    # ?val loss??????batch??
    if history["val_loss_per_epoch"]:
        val_x = history["val_batch_x"]
        val_losses = history["val_loss_per_epoch"]
        ax.plot(val_x, val_losses, 'o-', label="Val Loss", markersize=4)
    ax.set_xlabel("Batch")
    ax.set_ylabel("Loss")
    ax.set_title(f"Loss (up to Epoch {epoch+1})")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    # --- Acc ? ---
    fig, ax = plt.subplots(1, 1, figsize=(8, 4))
    ax.plot(batch_x, history["train_acc_per_batch"], label="Train Acc", alpha=0.6)
    if history["val_acc_per_epoch"]:
        val_x = history["val_batch_x"]
        val_accs = history["val_acc_per_epoch"]
        ax.plot(val_x, val_accs, 'o-', label="Val Acc", markersize=4)
    ax.set_xlabel("Batch")
    ax.set_ylabel("Accuracy")
    ax.set_title(f"Accuracy (up to Epoch {epoch+1})")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


def controller(seed: int,
               lstm_embed: int,
               hidden_dim: int,
               ratio: float,
               batch_size: int, 
               epochs: int, 
               learning_rate: float):
    torch.manual_seed(seed)
    data_path = Path(f"{DATAPATH}/processed_imdb_train_data.npz")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    model = LSTM(
            vocab_size=20000,
            embedding_dim=lstm_embed,
            hidden_dim=hidden_dim,
        ).to(device)
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    loss_fn = nn.CrossEntropyLoss()
    metric_fn = nn.CrossEntropyLoss()

    loader_dict = prepare_data_loader(data_path, ratio, batch_size)
    train_loader = loader_dict["train"]
    test_loader = loader_dict["test"]

    # ??????
    history = {
        "train_loss_per_batch": [],   # ??batch?train loss
        "train_acc_per_batch": [],    # ??batch?train acc
        "batch_indices": [],          # ??batch??
        "val_loss_per_epoch": [],     # ??eval?val loss
        "val_acc_per_epoch": [],      # ??eval?val acc
        "val_batch_x": [],            # val???batch x??(epoch????batch??)
    }

    global_batch_counter = 0
    start_time = time.time()

    for epoch in range(epochs):
        print(f"Epoch {epoch + 1} \n--------------------------------")
        train_loss, train_acc, global_batch_counter = train_per_epoch(
            model, loss_fn, optimizer, train_loader, device,
            epoch, start_time, global_batch_counter, history
        )
        # scheduler.step()

        val_loss, val_acc = test_per_epoch(model, metric_fn, test_loader, device)

        # ??val??(??epoch???,????)
        history["val_loss_per_epoch"].append(val_loss)
        history["val_acc_per_epoch"].append(val_acc)
        history["val_batch_x"].append(global_batch_counter)

        # ?5?epoch??val?????
        if (epoch + 1) % 5 == 0:
            elapsed = time.time() - start_time
            print(f"\n===== Epoch {epoch+1} Summary =====")
            print(f"Time: {elapsed:.1f}s")
            print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc*100:.1f}%")
            print(f"Val   Loss: {val_loss:.4f}, Val   Acc: {val_acc*100:.1f}%")
            print(f"================================\n")
            plot_metrics(history, epoch)

    print("Done!")
    return model

## `Console` Part
????????????,?????????????,???????????????,????????????

In [ ]:
# ?????:???????????????
# ???????????????,???????????????
# seed = ???????????
# ??:
seed = 12345678

# Transformer????????, ???LSTM??????
embed_dim = 32
num_heads = 2
ff_dim = 32
num_blocks = 2

# LSTM????????

lstm_embed = 16
hidden_dim = 16

ratio = 0.9 #??????????
batch_size = 1024 # batch???
epochs = 20 # ?????
learning_rate = 1e-3 #????????




model = controller(seed,
                   lstm_embed,
                   hidden_dim,
                   ratio,
                   batch_size,
                   epochs, 
                   learning_rate)

## `Evaluater` Part

<div class="alert alert-block alert-warning">
<b>??????????????,??????????????
</div>

????????????????,?????`kaggle`???????(`label`)??????,?????????????`submission.csv`??,?????`kaggle`?????,????????????????

In [ ]:
def evaluater(model: nn.Module):
    model.eval()
    test_data_path = Path("/kaggle/input/competitions/sjtu-2026-spring-lstm/shuffled-imdb_test_data.npz")
    submission_path = Path("/kaggle/working/submission.csv")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    with torch.no_grad():
        test_data = np.load(test_data_path)
        test_ids = test_data["ID"]
        test_tokens = test_data["x_test"]
        for i in range(5):
            batch_test_tokens = torch.tensor(test_tokens[5000*i: 5000*(i+1), :], dtype=torch.long).to(device)
            probs = model(batch_test_tokens)
            _, batch_predicted_labels = torch.max(probs, dim=-1)
            if i == 0:
                predicted_labels = batch_predicted_labels.cpu().numpy()
            else:
                predicted_labels = np.hstack((predicted_labels, batch_predicted_labels.cpu().numpy()))

    predicted_labels = pd.DataFrame(predicted_labels, index=test_ids, columns=["label"])
    predicted_labels.to_csv(submission_path, index=True, index_label="ID")
    print("evaluate Done")
evaluater(model)
